In [1]:
!apt-get update -qq && apt-get install -y -qq tesseract-ocr
!pip install -q transformers datasets evaluate accelerate pytesseract pillow scikit-learn

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


### Load Dataset & Create Binary Labels (medical vs non_medical)

In [2]:
import re
from datasets import load_dataset

# Load the dataset from Hugging Face
dataset = load_dataset("naazimsnh02/medocr-vision-dataset")
print("Dataset splits:", dataset)

# Keyword pattern to separate medical documents (1) from invoices/receipts (0)
MEDICAL_PATTERN = re.compile(
    r"(<s_ocr>|doctor_name|clinic_name|patient_name|medications|"
    r"hemoglobin|uhid|hospital|diagnosis|prescription|mg\b|ml\b|tablet|capsule|pathology|radiology)",
    re.IGNORECASE,
)


def assign_binary_label(example):
    text = example.get("text", "") or ""
    is_med = 1 if MEDICAL_PATTERN.search(text) else 0
    return {"label": is_med}


# Apply label mapping across train, validation, and test splits
dataset = dataset.map(assign_binary_label)

# Verify class balance
for split in dataset.keys():
    labels = dataset[split]["label"]
    print(
        f"{split}: Total={len(labels)}, Medical (1)={sum(labels)}, Non-Medical (0)={len(labels) - sum(labels)}"
    )


README.md:   0%|          | 0.00/9.86k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  411MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 52.8MB            

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 48.2MB            

... (5 outputs left out)

Map:   0%|          | 0/1969 [00:00<?, ? examples/s]

Map:   0%|          | 0/246 [00:00<?, ? examples/s]

Map:   0%|          | 0/247 [00:00<?, ? examples/s]

train: Total=1969, Medical (1)=1163, Non-Medical (0)=806
validation: Total=246, Medical (1)=132, Non-Medical (0)=114
test: Total=247, Medical (1)=145, Non-Medical (0)=102


### Preprocess Images with LayoutLMv3Processor

In [5]:
!pip install -q rapidocr_onnxruntime

In [6]:
import numpy as np
from PIL import Image
from rapidocr_onnxruntime import RapidOCR
from transformers import AutoProcessor

# 1. Initialize fast ONNX OCR engine (no subprocesses, no deadlocks)
ocr_engine = RapidOCR()

MODEL_ID = "microsoft/layoutlmv3-base"
# Set apply_ocr=False because we supply the words & normalized boxes directly
processor = AutoProcessor.from_pretrained(MODEL_ID, apply_ocr=False)


def extract_words_and_boxes(pil_image):
    w, h = pil_image.size
    img_np = np.array(pil_image)
    result, _ = ocr_engine(img_np)

    words, boxes = [], []
    if result:
        for line in result:
            coords, text, _ = line
            text = text.strip()
            if not text:
                continue
            # Convert 4-point polygon to normalized [0, 1000] bounding box
            xs = [pt[0] for pt in coords]
            ys = [pt[1] for pt in coords]
            box = [
                max(0, min(1000, int((min(xs) / w) * 1000))),
                max(0, min(1000, int((min(ys) / h) * 1000))),
                max(0, min(1000, int((max(xs) / w) * 1000))),
                max(0, min(1000, int((max(ys) / h) * 1000))),
            ]
            words.append(text)
            boxes.append(box)
    return words, boxes


def preprocess_batch(examples):
    images, all_words, all_boxes = [], [], []
    for img in examples["image"]:
        img = img.convert("RGB")
        img.thumbnail((1024, 1024))
        words, boxes = extract_words_and_boxes(img)
        images.append(img)
        all_words.append(words)
        all_boxes.append(boxes)

    encoding = processor(
        images,
        all_words,
        boxes=all_boxes,
        truncation=True,
        padding="max_length",
        max_length=512,
    )
    encoding["labels"] = examples["label"]
    return encoding


# 2. Run without num_proc so it updates smoothly every 16 images without forking deadlocks
encoded_dataset = dataset.map(
    preprocess_batch,
    batched=True,
    batch_size=16,
    remove_columns=dataset["train"].column_names,
)

encoded_dataset.set_format(type="torch")
print("Preprocessing complete:", encoded_dataset)


Parameter 'function'=<function preprocess_batch at 0x7964b1dc71a0> of the transform datasets.arrow_dataset.Dataset._map_single couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only shown once. Subsequent hashing failures won't be shown.


Map:   0%|          | 0/1969 [00:00<?, ? examples/s]

Map:   0%|          | 0/246 [00:00<?, ? examples/s]

Map:   0%|          | 0/247 [00:00<?, ? examples/s]

Preprocessing complete: DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'bbox', 'pixel_values', 'labels'],
        num_rows: 1969
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'bbox', 'pixel_values', 'labels'],
        num_rows: 246
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'bbox', 'pixel_values', 'labels'],
        num_rows: 247
    })
})


In [8]:
import json
import os

save_dir = "/kaggle/working/medocr_encoded_upload"
encoded_dataset.save_to_disk(f"{save_dir}/data")

metadata = {
    "title": "MedOCR LayoutLMv3 Encoded",
    "id": "rajdeeppa53/medocr-layoutlmv3-encoded",
    "licenses": [{"name": "CC0-1.0"}],
}

with open(f"{save_dir}/dataset-metadata.json", "w") as f:
    json.dump(metadata, f)

Saving the dataset (0/3 shards):   0%|          | 0/1969 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/246 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/247 [00:00<?, ? examples/s]

In [10]:
!kaggle datasets create -p /kaggle/working/medocr_encoded_upload --dir-mode zip

Starting upload for file data.zip
100%|████████████████████████████████████████| 220M/220M [00:02<00:00, 78.1MB/s]
Upload successful: data.zip (220MB)
Your private Dataset is being created. Please check progress at https://www.kaggle.com/datasets/rajdeeppa53/medocr-layoutlmv3-encoded


### Fine-Tune LayoutLMv3 on Kaggle GPU

In [11]:
import evaluate
import numpy as np
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments

# 1. Load accuracy and F1 metrics
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=predictions, references=labels)["accuracy"]
    f1 = f1_metric.compute(predictions=predictions, references=labels, average="binary")["f1"]
    return {"accuracy": round(acc, 4), "f1": round(f1, 4)}


# 2. Initialize LayoutLMv3 for binary classification
MODEL_ID = "microsoft/layoutlmv3-base"
id2label = {0: "non_medical", 1: "medical"}
label2id = {"non_medical": 0, "medical": 1}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
)

# 3. Configure training arguments optimized for Kaggle GPU
training_args = TrainingArguments(
    output_dir="./layoutlmv3-medical-gatekeeper",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    num_train_epochs=4,
    weight_decay=0.01,
    fp16=True,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=25,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["validation"],
    processing_class=processor,
    compute_metrics=compute_metrics,
)

# 4. Start training
trainer.train()


model.safetensors: reconstructing file:   0%|          |  0.00B /  501MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/212 [00:00<?, ?it/s]

[transformers] LayoutLMv3ForSequenceClassification LOAD REPORT from: microsoft/layoutlmv3-base
Key                        | Status  | 
---------------------------+---------+-
classifier.out_proj.bias   | MISSING | 
classifier.dense.weight    | MISSING | 
classifier.dense.bias      | MISSING | 
classifier.out_proj.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.629555,0.195181,0.975600,0.976900
2,0.491623,0.265380,0.979700,0.981000
3,0.403653,0.199496,0.983800,0.986100


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (2 outputs left out)

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


KeyboardInterrupt: 

### Evaluate on Test Set & Save/Zip the Model

In [14]:
import gc
import shutil
import torch

# 1. Free leftover VRAM from the interrupted training run
gc.collect()
torch.cuda.empty_cache()

# 2. Reduce eval batch size & offload logits to CPU periodically to prevent OOM
trainer.args.per_device_eval_batch_size = 4
trainer.args.eval_accumulation_steps = 10

# 3. Evaluate on the held-out test set
test_metrics = trainer.evaluate(encoded_dataset["test"])
print("=== Final Test Set Performance ===")
for key, value in test_metrics.items():
  print(f"{key}: {value:.4f}" if isinstance(value, float) else f"{key}: {value}")

# 4. Save the fine-tuned model weights and processor
SAVE_DIR = "/kaggle/working/medical_pdf_classifier"
trainer.save_model(SAVE_DIR)
processor.save_pretrained(SAVE_DIR)

# 5. Zip the saved directory
shutil.make_archive("/kaggle/working/medical_pdf_classifier", "zip", SAVE_DIR)
print(f"\nModel saved and zipped at: {SAVE_DIR}.zip")

=== Final Test Set Performance ===
eval_loss: 0.1995
eval_accuracy: 0.9838
eval_f1: 0.9861


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Model saved and zipped at: /kaggle/working/medical_pdf_classifier.zip


### Test Real PDF Inference (Digital & Scanned)

In [15]:
!pip install -q pymupdf

In [18]:
import io
import fitz  # PyMuPDF
import numpy as np
import torch
from PIL import Image
from rapidocr_onnxruntime import RapidOCR
from transformers import AutoModelForSequenceClassification, AutoProcessor

# 1. Load your saved fine-tuned model and processor
SAVE_DIR = "/kaggle/working/medical_pdf_classifier"
device = "cuda" if torch.cuda.is_available() else "cpu"

ocr_engine = RapidOCR()
loaded_processor = AutoProcessor.from_pretrained(SAVE_DIR, apply_ocr=False)
loaded_model = AutoModelForSequenceClassification.from_pretrained(SAVE_DIR).to(device)
loaded_model.eval()


def extract_words_and_boxes(pil_image):
    w, h = pil_image.size
    result, _ = ocr_engine(np.array(pil_image))
    words, boxes = [], []
    if result:
        for line in result:
            coords, text, _ = line
            text = text.strip()
            if not text:
                continue
            xs = [pt[0] for pt in coords]
            ys = [pt[1] for pt in coords]
            box = [
                max(0, min(1000, int((min(xs) / w) * 1000))),
                max(0, min(1000, int((min(ys) / h) * 1000))),
                max(0, min(1000, int((max(xs) / w) * 1000))),
                max(0, min(1000, int((max(ys) / h) * 1000))),
            ]
            words.append(text)
            boxes.append(box)
    return words, boxes


def predict_pdf_bytes(file_bytes: bytes, threshold: float = 0.85, max_pages: int = 2) -> dict:
    """Checks the first `max_pages` of a PDF. Approves if any page passes the medical threshold."""
    try:
        doc = fitz.open(stream=file_bytes, filetype="pdf")
    except Exception:
        return {"allowed": False, "confidence": 0.0, "reason": "Corrupted PDF"}

    if doc.is_encrypted or len(doc) == 0:
        return {"allowed": False, "confidence": 0.0, "reason": "Encrypted or empty PDF"}

    highest_score = 0.0
    pages_to_check = min(max_pages, len(doc))

    for page_idx in range(pages_to_check):
        pix = doc[page_idx].get_pixmap(dpi=150)
        img = Image.open(io.BytesIO(pix.tobytes("png"))).convert("RGB")
        img.thumbnail((1024, 1024))

        words, boxes = extract_words_and_boxes(img)
        inputs = loaded_processor(
            img,
            words,
            boxes=boxes,
            return_tensors="pt",
            truncation=True,
            max_length=512,
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            probs = torch.softmax(loaded_model(**inputs).logits, dim=-1)
            medical_prob = probs[0, 1].item()

        if medical_prob > highest_score:
            highest_score = medical_prob

        # Early exit if a medical page is already confirmed
        if highest_score >= threshold:
            break

    is_allowed = highest_score >= threshold
    return {
        "allowed": is_allowed,
        "confidence": round(highest_score, 4),
        "reason": "Verified medical document" if is_allowed else "Rejected: Non-medical PDF",
    }


# 2. Quick sanity test on 5 samples from the raw test split converted to PDF bytes
for idx in range(5):
    sample = dataset["test"][idx]
    buf = io.BytesIO()
    sample["image"].convert("RGB").save(buf, format="PDF")
    pdf_bytes = buf.getvalue()

    prediction = predict_pdf_bytes(pdf_bytes)
    actual_label = "medical" if sample["label"] == 1 else "non_medical"
    print(f"Sample {idx} | Actual: {actual_label:11s} | Prediction: {prediction}")


Loading weights:   0%|          | 0/216 [00:00<?, ?it/s]

Sample 0 | Actual: non_medical | Prediction: {'allowed': False, 'confidence': 0.0031, 'reason': 'Rejected: Non-medical PDF'}
Sample 1 | Actual: medical     | Prediction: {'allowed': True, 'confidence': 0.9996, 'reason': 'Verified medical document'}
Sample 2 | Actual: non_medical | Prediction: {'allowed': False, 'confidence': 0.0019, 'reason': 'Rejected: Non-medical PDF'}
Sample 3 | Actual: medical     | Prediction: {'allowed': True, 'confidence': 0.9929, 'reason': 'Verified medical document'}
Sample 4 | Actual: medical     | Prediction: {'allowed': True, 'confidence': 0.999, 'reason': 'Verified medical document'}
